# 🎭 Pun Translator — ML Evaluation Pipeline

**Fully offline** feature-rich SVM/GBM ensemble for pun detection, feeding Low's polygon algorithm for translation.

### Pipeline
| Step | Task | Method | Evaluator |
|------|------|--------|-----------|
| 1 | Identify pun word | GBM token classifier | `evaluate_pun_location` |
| 2 | Identify pun type | Phonetic heuristic + LR | `evaluate_pun_type` |
| 3 | Identify alternative meaning | Metaphone sound-alike lookup | `evaluate_alternative_words` |
| 4 | Translate pun → French/Spanish | Low's polygon algorithm | `evaluate_translations` |

### What you need
- `data/processed/combined_en_train.tsv` — the evaluation dataset
- `Lexique383.tsv` — download free from [lexique.org](http://lexique.org) (for Step 4 French homophones)
- No API keys. All offline.


## 0. Install & Imports

In [1]:
# All packages are free and offline after first download
!pip install -q pronouncing jellyfish scikit-learn pandas numpy scipy

import nltk
# WordNet and CMU dict — downloaded once, then cached
for resource in ['wordnet', 'omw-1.4', 'cmudict', 'averaged_perceptron_tagger']:
    nltk.download(resource, quiet=True)

print("✅ All dependencies ready")


✅ All dependencies ready


In [2]:
import re, os, json, difflib, collections, warnings
import pandas as pd
import numpy as np
import pronouncing
import jellyfish
import scipy.spatial.distance as spdist

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

from nltk.corpus import wordnet as wn
from typing import List, Optional, Tuple, Dict
from dataclasses import dataclass
import time

warnings.filterwarnings('ignore')

# ── Build CMU pronunciation dictionary ────────────────────────────────────────
pronouncing.init_cmu()
CMU: Dict[str, List[str]] = collections.defaultdict(list)
for _entry in pronouncing.pronunciations:
    _word = _entry[0].lower().rstrip('(0123456789)')
    CMU[_word].append(_entry[1])

# Metaphone index for fast sound-alike lookup
MPH_INDEX: Dict[str, set] = collections.defaultdict(set)
for _word in CMU:
    if _word.isalpha() and 2 < len(_word) < 20:
        MPH_INDEX[jellyfish.metaphone(_word)].add(_word)

print(f"✅ CMU dict: {len(CMU):,} words  |  Metaphone groups: {len(MPH_INDEX):,}")


✅ CMU dict: 126,052 words  |  Metaphone groups: 47,784


## 1. Configuration

In [3]:
# ── USER CONFIG ─────────────────────────────────────────────────────────────
TSV_PATH     = "combined_en_train.tsv"
LEXIQUE_PATH = "/content/Lexique383.tsv"   # ← update to your local path
TARGET_LANG  = "fr"                        # "fr" or "es"
OUTPUT_DIR   = "outputs/"
SAMPLE_N     = 100    # rows to use for quick evaluation demo (None = all)
RANDOM_SEED  = 42
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config: lang={TARGET_LANG}, sample_n={SAMPLE_N}, output={OUTPUT_DIR}")


✅ Config: lang=fr, sample_n=100, output=outputs/


## 2. Load Dataset

In [4]:
df_all = pd.read_csv(TSV_PATH, sep="\t")
df_all = df_all.dropna(subset=["text_clean","manual_location","manual_type","manual_alternative"])
df_all = df_all.astype({
    "text_clean": str, "manual_location": str,
    "manual_type": str, "manual_alternative": str
})
print(f"Total rows: {len(df_all)}")
print(f"Pun type distribution:\n{df_all['manual_type'].value_counts().to_string()}")
print()

# Sample for evaluation demo
if SAMPLE_N and SAMPLE_N < len(df_all):
    df_eval = df_all.sample(SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Using {SAMPLE_N}-row sample for evaluation demo.")
else:
    df_eval = df_all.reset_index(drop=True)
    print("Using full dataset.")

df_eval.head(3)


Total rows: 1404
Pun type distribution:
manual_type
homographic    730
homophonic     674

Using 100-row sample for evaluation demo.


,id_en,text_clean,manual_location,manual_type,manual_alternative
0,en_6789,There was a sale at the fish market today. I w...,catch,homographic,catch
1,en_3677,She was suspected of stealing a brooch but the...,pin,homographic,pin
2,en_4418,When asked by her co-workers whether they shou...,presence,homophonic,presents


## 3. Shared Utilities

In [5]:
STOPWORDS = set(
    "the a an and or but if then else to of in on at for with is are was were be "
    "been being it this that these those i you he she we they me him her us them "
    "my your his their as by from not no so just only very really there here than "
    "when what which who how".split()
)

def tokenize(s: str) -> List[str]:
    return re.findall(r"[A-Za-z]+", s.lower())

def content_tokens(s: str) -> List[Tuple[str, int]]:
    """Return (token, original_index) for non-stopword tokens of length > 2."""
    toks = tokenize(s)
    return [(t, i) for i, t in enumerate(toks) if t not in STOPWORDS and len(t) > 2]

def n_syllables(word: str) -> int:
    phones = CMU.get(word.lower(), [])
    return len([p for p in phones[0].split() if p[-1].isdigit()]) if phones else 0

def n_soundalikes(word: str) -> int:
    mph = jellyfish.metaphone(word.lower())
    return max(0, len(MPH_INDEX.get(mph, set())) - 1)

def soundalike_words(word: str, limit: int = 20) -> List[str]:
    mph = jellyfish.metaphone(word.lower())
    return [w for w in MPH_INDEX.get(mph, set()) if w != word.lower()][:limit]

def wn_n_senses(word: str) -> int:
    """Number of WordNet synsets for a word (polysemy proxy)."""
    try:
        return len(wn.synsets(word))
    except Exception:
        return 0

def wn_max_sense_distance(word: str) -> float:
    """Max shortest-path distance between any two senses (semantic spread)."""
    try:
        ss = wn.synsets(word)
        if len(ss) < 2:
            return 0.0
        best = 0.0
        for i in range(min(len(ss), 6)):
            for j in range(i+1, min(len(ss), 6)):
                d = ss[i].shortest_path_distance(ss[j])
                if d is not None and d > best:
                    best = float(d)
        return best
    except Exception:
        return 0.0

print("✅ Utilities loaded")


✅ Utilities loaded


## 4. Step 1 — Pun Word Identification

### Model
- **Task**: token classification — for every content word in a sentence, predict
  whether it is the pun word (binary label)
- **Prediction**: take the token with the highest predicted probability per sentence
- **Features**: position, phonetics (CMU), WordNet polysemy, rhyme count, OOV flag
- **Model**: Gradient Boosting (GBM) — best performer on SemEval-style pun tasks
  at this dataset size


In [6]:
FEAT_NAMES_S1 = [
    'rel_pos',        # relative position in sentence (0=start, 1=end)
    'is_last',        # is the last content token
    'is_near_last',   # is one of last 2 content tokens
    'in_last_qtr',    # in last 25% of sentence
    'in_last_tenth',  # in last 10% of sentence
    'wlen',           # word length
    'in_cmu',         # in CMU pronunciation dict
    'is_oov',         # out-of-vocabulary (homophonic pun proxy)
    'nsyl',           # syllable count
    'n_soundalikes',  # # of words with same metaphone (homophonic potential)
    'n_rhymes',       # rhyme count
    'metaphone_len',  # length of metaphone representation
    'wn_n_senses',    # WordNet polysemy (# synsets)
    'wn_sense_dist',  # max semantic spread across senses
    'after_comma',    # appears after a comma (punchline signal)
    'after_semicolon',
    'repeat',         # frequency of this word in the sentence
    'wlen_gt5',
    'wlen_gt8',
    'pun_suffix',     # ends with common pun-carrier suffixes (er/or/ing/ed/ly)
    'in_second_half', # in second half of sentence
]

def extract_features_s1(tok: str, idx: int, all_toks: List[str], sent: str) -> List[float]:
    w      = tok.lower()
    n      = len(all_toks)
    rp     = idx / max(n - 1, 1)
    in_cmu = int(w in CMU)
    nsyl   = n_syllables(w)
    n_sa   = n_soundalikes(w)
    n_rh   = min(len(pronouncing.rhymes(w)), 60)
    mph    = jellyfish.metaphone(w)
    sl     = sent.lower()
    tp     = sl.find(w)
    n_wn   = wn_n_senses(w)
    wn_sd  = wn_max_sense_distance(w)

    return [
        rp,
        int(idx == n - 1),
        int(idx >= n - 2),
        int(rp >= 0.75),
        int(rp >= 0.90),
        len(w),
        in_cmu,
        int(not in_cmu),
        nsyl,
        n_sa,
        n_rh,
        len(mph),
        n_wn,
        wn_sd,
        int(tp > 0 and ',' in sl[:tp]),
        int(tp > 0 and ';' in sl[:tp]),
        sum(1 for t in all_toks if t == w),
        int(len(w) > 5),
        int(len(w) > 8),
        int(w.endswith(('er','or','ing','ed','ly'))),
        int(rp >= 0.5),
    ]

print("✅ Step 1 feature extractor defined")
print(f"   Feature count: {len(FEAT_NAMES_S1)}")


✅ Step 1 feature extractor defined
   Feature count: 21


In [7]:
print("Building token-level training data from full dataset...")

X_rows, y_arr, groups_arr, meta_rows = [], [], [], []

for row_idx, row in df_all.iterrows():
    toks     = tokenize(row['text_clean'])
    pun_loc  = row['manual_location'].lower().strip()
    pun_idxs = [i for i, t in enumerate(toks) if t == pun_loc]
    pun_idx  = max(pun_idxs) if pun_idxs else -1

    for i, tok in enumerate(toks):
        if tok in STOPWORDS or len(tok) <= 2:
            continue
        X_rows.append(extract_features_s1(tok, i, toks, row['text_clean']))
        y_arr.append(1 if i == pun_idx else 0)
        groups_arr.append(row_idx)
        meta_rows.append({
            'id_en': row['id_en'],
            'tok': tok,
            'tok_idx': i,
            'sentence': row['text_clean'],
            'manual_location': row['manual_location'],
        })

X_s1 = np.array(X_rows, dtype=float)
y_s1 = np.array(y_arr)
groups_s1 = np.array(groups_arr)
meta_df_s1 = pd.DataFrame(meta_rows)

print(f"Token candidates: {len(y_s1):,}")
print(f"Positive labels (pun words): {y_s1.sum():,}  ({y_s1.mean():.1%} of candidates)")

# ── Train GBM on full dataset ──────────────────────────────────────────────────
clf_s1 = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=5,
    random_state=RANDOM_SEED,
)
clf_s1.fit(X_s1, y_s1)
print("\n✅ GBM trained on full dataset")

# ── Feature importance ─────────────────────────────────────────────────────────
fi = sorted(zip(FEAT_NAMES_S1, clf_s1.feature_importances_), key=lambda x: -x[1])
print("\nTop feature importances:")
for fname, fval in fi[:10]:
    bar = '█' * int(fval * 80)
    print(f"  {fname:<20} {fval:.4f}  {bar}")


Building token-level training data from full dataset...
Token candidates: 9,473
Positive labels (pun words): 1,391  (14.7% of candidates)

✅ GBM trained on full dataset

Top feature importances:
  rel_pos              0.3033  ████████████████████████
  is_near_last         0.2812  ██████████████████████
  wn_n_senses          0.0937  ███████
  wn_sense_dist        0.0890  ███████
  n_soundalikes        0.0693  █████
  wlen                 0.0518  ████
  n_rhymes             0.0288  ██
  pun_suffix           0.0154  █
  nsyl                 0.0151  █
  after_comma          0.0111  


In [8]:
# ── Cross-validated evaluation (group-aware: whole sentences in train OR test) ──
print("Running group-aware cross-validation (5 folds)...")

gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=RANDOM_SEED)

fold_results = []
for fold, (train_idx, test_idx) in enumerate(gss.split(X_s1, y_s1, groups_s1)):
    clf_cv = GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=5, random_state=RANDOM_SEED
    )
    clf_cv.fit(X_s1[train_idx], y_s1[train_idx])

    # Token-level
    pred_tok = clf_cv.predict(X_s1[test_idx])
    tok_f1   = f1_score(y_s1[test_idx], pred_tok, zero_division=0)

    # Sentence-level (argmax proba per sentence)
    proba_test = clf_cv.predict_proba(X_s1[test_idx])[:, 1]
    meta_test  = meta_df_s1.iloc[test_idx].copy()
    meta_test['proba'] = proba_test
    meta_test['label'] = y_s1[test_idx]

    sent_correct = 0
    test_sids = meta_test['id_en'].unique()
    for sid in test_sids:
        grp = meta_test[meta_test['id_en'] == sid]
        best = grp.loc[grp['proba'].idxmax()]
        if best['label'] == 1:
            sent_correct += 1

    sent_acc = sent_correct / len(test_sids)
    fold_results.append({'fold': fold+1, 'tok_f1': tok_f1, 'sent_acc': sent_acc})
    print(f"  Fold {fold+1}: token F1={tok_f1:.3f}  sentence acc={sent_acc:.3f}")

fr = pd.DataFrame(fold_results)
print(f"\n{'─'*55}")
print(f"  Mean token-level F1      : {fr['tok_f1'].mean():.3f} ± {fr['tok_f1'].std():.3f}")
print(f"  Mean sentence-level acc  : {fr['sent_acc'].mean():.3f} ± {fr['sent_acc'].std():.3f}")
print(f"{'─'*55}")


Running group-aware cross-validation (5 folds)...
  Fold 1: token F1=0.685  sentence acc=0.737
  Fold 2: token F1=0.647  sentence acc=0.698
  Fold 3: token F1=0.641  sentence acc=0.694
  Fold 4: token F1=0.588  sentence acc=0.673
  Fold 5: token F1=0.676  sentence acc=0.762

───────────────────────────────────────────────────────
  Mean token-level F1      : 0.647 ± 0.038
  Mean sentence-level acc  : 0.712 ± 0.036
───────────────────────────────────────────────────────


In [9]:
# ── Run predictions on the evaluation sample ──────────────────────────────────
print(f"Running Step 1 predictions on {len(df_eval)}-row eval sample...")

def predict_pun_location(sentence: str) -> str:
    """Return the predicted pun word for one sentence."""
    toks = tokenize(sentence)
    candidates = []
    for i, tok in enumerate(toks):
        if tok in STOPWORDS or len(tok) <= 2:
            continue
        feats = extract_features_s1(tok, i, toks, sentence)
        candidates.append((tok, feats))
    if not candidates:
        return ""
    X_cand = np.array([c[1] for c in candidates], dtype=float)
    probas  = clf_s1.predict_proba(X_cand)[:, 1]
    best_i  = int(np.argmax(probas))
    return candidates[best_i][0]

df_step1 = df_eval[['id_en','text_clean','manual_location']].copy()
df_step1['predicted_location'] = df_step1['text_clean'].apply(predict_pun_location)

# ── evaluator.py — evaluate_pun_location ──────────────────────────────────────
def evaluate_pun_location(df: pd.DataFrame) -> dict:
    """
    Mirrors evaluator.py evaluate_pun_location.
    Expects columns: predicted_location, manual_location.
    """
    y_true = df['manual_location'].str.lower().str.strip()
    y_pred = df['predicted_location'].str.lower().str.strip()

    # Binary: correct if exact match
    correct = (y_true == y_pred).astype(int)
    total   = len(df)
    tp      = correct.sum()

    acc  = tp / total
    # Treat as binary classification: positive = "matched correctly"
    prec = precision_score(correct, correct, zero_division=0)  # always 1 when using correct
    # Real precision/recall: predicted positive = any non-empty prediction
    y_true_bin = [1] * total
    y_pred_bin = correct.tolist()
    prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    rec  = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    f1   = f1_score(y_true_bin, y_pred_bin, zero_division=0)

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'correct': int(tp), 'total': total}

metrics_s1 = evaluate_pun_location(df_step1)

print("\n" + "═"*55)
print("STEP 1 RESULTS — Pun Word Identification")
print("═"*55)
print(f"  Accuracy  : {metrics_s1['accuracy']:.3f}  ({metrics_s1['correct']}/{metrics_s1['total']})")
print(f"  Precision : {metrics_s1['precision']:.3f}")
print(f"  Recall    : {metrics_s1['recall']:.3f}")
print(f"  F1        : {metrics_s1['f1']:.3f}")
print("═"*55)

# Save output TSV
step1_out = os.path.join(OUTPUT_DIR, "step1_pun_location.tsv")
df_step1.to_csv(step1_out, sep="\t", index=False)
print(f"\n  Saved → {step1_out}")

# Sample predictions
print("\nSample predictions (first 10):")
print(f"  {'Sentence':<55} {'Predicted':<15} {'Ground Truth':<15} {'✓'}")
print(f"  {'─'*55} {'─'*15} {'─'*15} {'─'*2}")
for _, r in df_step1.head(10).iterrows():
    ok = '✅' if r['predicted_location'].lower() == r['manual_location'].lower() else '❌'
    sent = r['text_clean'][:52] + '...' if len(r['text_clean']) > 52 else r['text_clean']
    print(f"  {sent:<55} {r['predicted_location']:<15} {r['manual_location']:<15} {ok}")


Running Step 1 predictions on 100-row eval sample...

═══════════════════════════════════════════════════════
STEP 1 RESULTS — Pun Word Identification
═══════════════════════════════════════════════════════
  Accuracy  : 0.900  (90/100)
  Precision : 1.000
  Recall    : 0.900
  F1        : 0.947
═══════════════════════════════════════════════════════

  Saved → outputs/step1_pun_location.tsv

Sample predictions (first 10):
  Sentence                                                Predicted       Ground Truth    ✓
  ─────────────────────────────────────────────────────── ─────────────── ─────────────── ──
  There was a sale at the fish market today. I went to... catch           catch           ✅
  She was suspected of stealing a brooch but they coul... pin             pin             ✅
  When asked by her co-workers whether they should bri... presence        presence        ✅
  I want to buy that big diamond, he said hopefully.      hopefully       hopefully       ✅
  Camille relocated 

## 5. Step 2 — Pun Type Identification (Optional)

**Homographic**: same spelling, different meaning (e.g. *bank* the river vs *bank* the institution).
**Homophonic**: different spelling, same/similar sound (e.g. *grater* / *greater*).

### Key insight
- If `metaphone(pun_word) == metaphone(alternative)` → homophonic
- If the pun word is in WordNet with many senses → homographic
- A logistic regression on these features generalises this cleanly


In [10]:
def extract_features_s2(pun_word: str, sentence: str) -> List[float]:
    """Features for pun type classification."""
    w      = pun_word.lower().strip()
    in_cmu = int(w in CMU)
    n_sa   = n_soundalikes(w)
    n_wn   = wn_n_senses(w)
    wn_sd  = wn_max_sense_distance(w)

    # Phonetic complexity: longer metaphone = more distinctive sound signature
    mph_len = len(jellyfish.metaphone(w))

    # Does sentence contain near-homophones of the pun word?
    toks = tokenize(sentence)
    mph_w = jellyfish.metaphone(w)
    n_mph_matches_in_sent = sum(
        1 for t in toks if t != w and jellyfish.metaphone(t) == mph_w
    )

    return [
        in_cmu,          # OOV words are almost always homophonic puns
        int(not in_cmu),
        n_sa,            # many sound-alikes → more likely homophonic
        n_wn,            # many WordNet senses → more likely homographic
        wn_sd,           # wide sense spread → homographic
        mph_len,
        n_mph_matches_in_sent,
        len(w),
    ]

FEAT_NAMES_S2 = ['in_cmu','is_oov','n_soundalikes','wn_n_senses',
                  'wn_sense_dist','metaphone_len','n_mph_in_sent','wlen']

# Build training data for Step 2 using full dataset + Step 1 pun word detections
# (use ground-truth location for training, predicted for evaluation)
X_s2_rows, y_s2_arr = [], []
le_type = LabelEncoder()

for _, row in df_all.iterrows():
    feats = extract_features_s2(row['manual_location'], row['text_clean'])
    X_s2_rows.append(feats)
    y_s2_arr.append(row['manual_type'])

X_s2 = np.array(X_s2_rows, dtype=float)
y_s2 = le_type.fit_transform(y_s2_arr)  # homographic=0, homophonic=1 (alphabetical)
print(f"Step 2 classes: {le_type.classes_}")

clf_s2 = LogisticRegression(C=1.0, max_iter=500, random_state=RANDOM_SEED)
clf_s2.fit(X_s2, y_s2)

# Cross-val
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
from sklearn.model_selection import cross_val_score
cv_f1 = cross_val_score(clf_s2, X_s2, y_s2, cv=skf, scoring='f1_macro')
print(f"Step 2 CV macro-F1: {cv_f1.mean():.3f} ± {cv_f1.std():.3f}")

print("✅ Step 2 classifier trained")


Step 2 classes: ['homographic' 'homophonic']
Step 2 CV macro-F1: 0.775 ± 0.017
✅ Step 2 classifier trained


In [11]:
# ── Run Step 2 on eval sample ──────────────────────────────────────────────────
def predict_pun_type(pun_word: str, sentence: str) -> str:
    if not pun_word:
        return 'homographic'
    feats = extract_features_s2(pun_word, sentence)
    pred  = clf_s2.predict(np.array([feats]))[0]
    return le_type.inverse_transform([pred])[0]

df_step2 = df_eval[['id_en','text_clean','manual_type']].copy()
df_step2['predicted_type'] = df_step2.apply(
    lambda r: predict_pun_type(
        df_step1.loc[df_step1['id_en']==r['id_en'], 'predicted_location'].values[0]
        if r['id_en'] in df_step1['id_en'].values else '',
        r['text_clean']
    ), axis=1
)

def evaluate_pun_type(df: pd.DataFrame) -> dict:
    """Mirrors evaluator.py evaluate_pun_type."""
    y_true = df['manual_type'].str.strip()
    y_pred = df['predicted_type'].str.strip()
    labels = sorted(y_true.unique())
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

metrics_s2 = evaluate_pun_type(df_step2)

print("\n" + "═"*55)
print("STEP 2 RESULTS — Pun Type Classification")
print("═"*55)
for k, v in metrics_s2.items():
    print(f"  {k.capitalize():<12}: {v:.3f}")
print("═"*55)

print("\nDetailed report:")
print(classification_report(df_step2['manual_type'], df_step2['predicted_type']))

step2_out = os.path.join(OUTPUT_DIR, "step2_pun_type.tsv")
df_step2.to_csv(step2_out, sep="\t", index=False)
print(f"Saved → {step2_out}")



═══════════════════════════════════════════════════════
STEP 2 RESULTS — Pun Type Classification
═══════════════════════════════════════════════════════
  Accuracy    : 0.760
  Precision   : 0.764
  Recall      : 0.760
  F1          : 0.759
═══════════════════════════════════════════════════════

Detailed report:
              precision    recall  f1-score   support

 homographic       0.80      0.70      0.74        50
  homophonic       0.73      0.82      0.77        50

    accuracy                           0.76       100
   macro avg       0.76      0.76      0.76       100
weighted avg       0.76      0.76      0.76       100

Saved → outputs/step2_pun_type.tsv


## 6. Step 3 — Alternative Meaning Identification (Optional)

**Strategy**:
- **Homophonic puns**: find words with the same Metaphone code as the pun word
  (they sound alike but spell differently). Rank candidates by frequency in CMU
  and pick the most plausible.
- **Homographic puns**: the alternative IS the pun word (same spelling, different
  sense), so return it unchanged.


In [12]:
def exact_homophone_words(word: str, limit: int = 50) -> List[str]:
    """
    Return words that share an exact CMU pronunciation with `word`.
    This is much stricter than Metaphone and fixes errors like whey -> whoa,
    because whey and way are exact CMU homophones, while whoa is not.
    """
    w = word.lower().strip()
    phones = CMU.get(w, [])
    if not phones:
        return []

    out = []
    seen = {w}
    target_phones = set(phones)

    for cand, cand_phones in CMU.items():
        if cand in seen or not cand.isalpha():
            continue
        if target_phones.intersection(cand_phones):
            out.append(cand)
            seen.add(cand)
            if len(out) >= limit:
                break
    return out


def min_phone_edit_distance(w1: str, w2: str) -> int:
    """
    Minimum Levenshtein distance between any CMU pronunciation pair.
    Lower is better. 0 means exact homophone.
    """
    p1s = CMU.get(w1.lower().strip(), [])
    p2s = CMU.get(w2.lower().strip(), [])
    if not p1s or not p2s:
        return 999

    best = 999
    for p1 in p1s:
        for p2 in p2s:
            d = jellyfish.levenshtein_distance(p1, p2)
            if d < best:
                best = d
    return best


def candidate_semantic_fit(candidate: str, sentence: str, pun_word: str) -> float:
    """
    Very lightweight contextual score:
    prefer candidates that have WordNet senses and some semantic relation
    to nearby content words in the sentence.
    """
    cand = candidate.lower().strip()
    toks = [t for t in tokenize(sentence) if t != pun_word.lower().strip()]
    ctx  = [t for t in toks if t not in STOPWORDS and len(t) > 2][:8]

    score = 0.0

    # More WordNet senses usually means more plausible lexical item
    score += 0.15 * wn_n_senses(cand)

    # Mild bonus for semantic relatedness to context tokens
    try:
        cand_syns = wn.synsets(cand)
        if cand_syns and ctx:
            best_ctx = 0.0
            for tok in ctx:
                tok_syns = wn.synsets(tok)
                for s1 in cand_syns[:4]:
                    for s2 in tok_syns[:4]:
                        sim = s1.wup_similarity(s2)
                        if sim and sim > best_ctx:
                            best_ctx = sim
            score += best_ctx
    except Exception:
        pass

    return score


def score_alternative_candidate(candidate: str, pun_word: str, sentence: str) -> Tuple:
    """
    Lower / higher tuple ranking:
    1) exact CMU homophone first
    2) smaller phone edit distance
    3) closer word length
    4) more WordNet senses
    5) better sentence-context semantic fit
    """
    c = candidate.lower().strip()
    w = pun_word.lower().strip()

    phone_dist = min_phone_edit_distance(w, c)
    exact_homo = int(phone_dist == 0)

    return (
        -exact_homo,                              # exact homophones first
        phone_dist,                              # then closest pronunciation
        abs(len(c) - len(w)),                    # then similar length
        -wn_n_senses(c),                         # then lexical plausibility
        -candidate_semantic_fit(c, sentence, w)  # then context fit
    )


def predict_alternative(pun_word: str, pun_type: str, sentence: str) -> str:
    """
    Predict the alternative spelling/word for the pun.

    Homographic:
        same surface form, different meaning -> return pun word.
    Homophonic:
        prefer exact CMU homophones;
        fall back to broader Metaphone neighbors only if needed.
    """
    if not pun_word:
        return ''

    w = pun_word.lower().strip()

    if pun_type == 'homographic':
        return w

    # Tier 1: exact CMU homophones
    exact_alts = exact_homophone_words(w, limit=50)

    # Tier 2: broader Metaphone neighbors if no exact homophones found
    broad_alts = soundalike_words(w, limit=50)

    # Merge while preserving order and excluding self
    merged = []
    seen = {w}
    for group in [exact_alts, broad_alts]:
        for cand in group:
            cand = cand.lower().strip()
            if not cand or cand in seen:
                continue
            if not cand.isalpha():
                continue
            merged.append(cand)
            seen.add(cand)

    if not merged:
        return w

    ranked = sorted(merged, key=lambda c: score_alternative_candidate(c, w, sentence))
    return ranked[0]


df_step3 = df_eval[['id_en','text_clean','manual_alternative']].copy()
df_step3['predicted_location'] = df_step1['predicted_location'].values
df_step3['predicted_type']     = df_step2['predicted_type'].values
df_step3['predicted_alternative'] = df_step3.apply(
    lambda r: predict_alternative(
        r['predicted_location'],
        r['predicted_type'],
        r['text_clean']
    ),
    axis=1
)

def evaluate_alternative_words(df: pd.DataFrame) -> dict:
    """Mirrors evaluator.py evaluate_alternative_words."""
    y_true = df['manual_alternative'].str.lower().str.strip()
    y_pred = df['predicted_alternative'].str.lower().str.strip()
    correct = (y_true == y_pred).astype(int)
    total   = len(df)
    tp      = correct.sum()
    acc     = tp / total
    prec    = precision_score([1]*total, correct.tolist(), zero_division=0)
    rec     = recall_score([1]*total, correct.tolist(), zero_division=0)
    f1      = f1_score([1]*total, correct.tolist(), zero_division=0)
    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'correct': int(tp),
        'total': total
    }

metrics_s3 = evaluate_alternative_words(df_step3)

print("\n" + "═"*55)
print("STEP 3 RESULTS — Alternative Meaning Identification")
print("═"*55)
print(f"  Accuracy  : {metrics_s3['accuracy']:.3f}  ({metrics_s3['correct']}/{metrics_s3['total']})")
print(f"  Precision : {metrics_s3['precision']:.3f}")
print(f"  Recall    : {metrics_s3['recall']:.3f}")
print(f"  F1        : {metrics_s3['f1']:.3f}")
print("═"*55)

for ptype in ['homographic','homophonic']:
    mask = df_step3['predicted_type'] == ptype
    sub  = df_step3[mask]
    if len(sub) == 0:
        continue
    correct_sub = (
        sub['manual_alternative'].str.lower().str.strip() ==
        sub['predicted_alternative'].str.lower().str.strip()
    ).sum()
    print(f"  {ptype}: {correct_sub}/{len(sub)} = {correct_sub/len(sub):.2%}")

step3_out = os.path.join(OUTPUT_DIR, "step3_alternative.tsv")
df_step3[['id_en','text_clean','manual_alternative','predicted_alternative',
           'predicted_location','predicted_type']].to_csv(step3_out, sep="\t", index=False)
print(f"\nSaved → {step3_out}")

# def predict_alternative(pun_word: str, pun_type: str) -> str:
#     """
#     Predict the alternative spelling/word for the pun.
#     For homophonic: find best sound-alike. For homographic: return pun word.
#     """
#     if not pun_word:
#         return ''
#     w = pun_word.lower().strip()

#     if pun_type == 'homographic':
#         return w  # same surface form, different sense

#     # Homophonic: find sound-alikes via Metaphone
#     alts = soundalike_words(w, limit=30)
#     if not alts:
#         return w  # fallback

#     # Rank: prefer words of similar length, exclude the word itself
#     alts_scored = sorted(alts, key=lambda c: (
#         abs(len(c) - len(w)),      # prefer similar length
#         -wn_n_senses(c),           # prefer words with WordNet senses
#     ))
#     return alts_scored[0] if alts_scored else w

# df_step3 = df_eval[['id_en','text_clean','manual_alternative']].copy()
# df_step3['predicted_location'] = df_step1['predicted_location'].values
# df_step3['predicted_type']     = df_step2['predicted_type'].values
# df_step3['predicted_alternative'] = df_step3.apply(
#     lambda r: predict_alternative(r['predicted_location'], r['predicted_type']), axis=1
# )

# def evaluate_alternative_words(df: pd.DataFrame) -> dict:
#     """Mirrors evaluator.py evaluate_alternative_words."""
#     y_true = df['manual_alternative'].str.lower().str.strip()
#     y_pred = df['predicted_alternative'].str.lower().str.strip()
#     correct = (y_true == y_pred).astype(int)
#     total   = len(df)
#     tp      = correct.sum()
#     acc     = tp / total
#     prec    = precision_score([1]*total, correct.tolist(), zero_division=0)
#     rec     = recall_score([1]*total, correct.tolist(), zero_division=0)
#     f1      = f1_score([1]*total, correct.tolist(), zero_division=0)
#     return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
#             'correct': int(tp), 'total': total}

# metrics_s3 = evaluate_alternative_words(df_step3)

# print("\n" + "═"*55)
# print("STEP 3 RESULTS — Alternative Meaning Identification")
# print("═"*55)
# print(f"  Accuracy  : {metrics_s3['accuracy']:.3f}  ({metrics_s3['correct']}/{metrics_s3['total']})")
# print(f"  Precision : {metrics_s3['precision']:.3f}")
# print(f"  Recall    : {metrics_s3['recall']:.3f}")
# print(f"  F1        : {metrics_s3['f1']:.3f}")
# print("═"*55)

# # Split by pun type
# for ptype in ['homographic','homophonic']:
#     mask = df_step3['predicted_type'] == ptype
#     sub  = df_step3[mask]
#     if len(sub) == 0: continue
#     correct_sub = (sub['manual_alternative'].str.lower() == sub['predicted_alternative'].str.lower()).sum()
#     print(f"  {ptype}: {correct_sub}/{len(sub)} = {correct_sub/len(sub):.2%}")

# step3_out = os.path.join(OUTPUT_DIR, "step3_alternative.tsv")
# df_step3[['id_en','text_clean','manual_alternative','predicted_alternative',
#            'predicted_location','predicted_type']].to_csv(step3_out, sep="\t", index=False)
# print(f"\nSaved → {step3_out}")



═══════════════════════════════════════════════════════
STEP 3 RESULTS — Alternative Meaning Identification
═══════════════════════════════════════════════════════
  Accuracy  : 0.450  (45/100)
  Precision : 1.000
  Recall    : 0.450
  F1        : 0.621
═══════════════════════════════════════════════════════
  homographic: 31/44 = 70.45%
  homophonic: 14/56 = 25.00%

Saved → outputs/step3_alternative.tsv


## 7. Step 4 — Low's Polygon Translation Engine

The pun word and its alternative are fed into **Low's polygon algorithm**, which
searches progressively longer semantic/phonetic paths in the target language to
find a word that simultaneously carries both meanings of the pun.

Requires:
- `Lexique383.tsv` for French phonetic lookup (download from lexique.org)
- `argostranslate` for fully-offline word translation (no API key)

```
pip install argostranslate
```
Then in Python: download the en→fr or en→es language pack once.


In [13]:
pip install argostranslate

In [14]:
# ── Offline translator via argostranslate ─────────────────────────────────────
from argostranslate import package, translate

def _argos_pair_installed(src: str, tgt: str) -> bool:
    """
    Check installed Argos packages without relying on nonexistent
    AvailablePackage.is_installed().
    """
    try:
        installed_pkgs = package.get_installed_packages()
    except Exception:
        installed_pkgs = []

    for pkg in installed_pkgs:
        from_code = getattr(pkg, "from_code", None)
        to_code   = getattr(pkg, "to_code", None)
        if from_code == src and to_code == tgt:
            return True
    return False


def setup_argos(src: str = 'en', tgt: str = 'fr'):
    """
    Download/install the language pack once, then return a translator object.
    Uses the documented Argos package APIs.
    """
    package.update_package_index()

    if not _argos_pair_installed(src, tgt):
        available = package.get_available_packages()
        pkg = next((p for p in available if p.from_code == src and p.to_code == tgt), None)

        if pkg is None:
            # Try the built-in installer for language pairs as a fallback.
            ok = package.install_package_for_language_pair(src, tgt)
            if not ok:
                raise ValueError(f"No argostranslate package found for {src}→{tgt}")
        else:
            print(f"Downloading {src}→{tgt} language pack…")
            download_path = pkg.download()
            package.install_from_path(download_path)
            print("Done.")

    installed_langs = translate.get_installed_languages()
    src_lang = next((l for l in installed_langs if l.code == src), None)
    tgt_lang = next((l for l in installed_langs if l.code == tgt), None)

    if src_lang is None or tgt_lang is None:
        raise RuntimeError(f"Installed Argos languages missing for {src}→{tgt}")

    return src_lang.get_translation(tgt_lang)


_ARGOS_TRANSLATOR = None
_ARGOS_TRANSLATOR_PAIR = None


def argos_translate(text: str, src: str = 'en', tgt: str = TARGET_LANG) -> str:
    global _ARGOS_TRANSLATOR, _ARGOS_TRANSLATOR_PAIR

    text = "" if text is None else str(text).strip()
    if not text:
        return ""

    if _ARGOS_TRANSLATOR is None or _ARGOS_TRANSLATOR_PAIR != (src, tgt):
        _ARGOS_TRANSLATOR = setup_argos(src, tgt)
        _ARGOS_TRANSLATOR_PAIR = (src, tgt)

    return _ARGOS_TRANSLATOR.translate(text)


print("✅ Argostranslate wrapper ready")

✅ Argostranslate wrapper ready


In [15]:
# ── Data models ───────────────────────────────────────────────────────────────
@dataclass
class TranslationCandidate:
    pun_word: str
    polygon_level: int
    path: List[str]
    explanation: str
    confidence: float

@dataclass
class FallbackTranslation:
    strategy: str
    translation: str
    explanation: str


# ── Translation cache ──────────────────────────────────────────────────────────
_TRANS_CACHE: Dict[str, str] = {}

def translate_word(word: str) -> str:
    w = "" if word is None else str(word).strip().lower()
    if not w:
        return ""
    if w in _TRANS_CACHE:
        return _TRANS_CACHE[w]
    result = argos_translate(w)
    _TRANS_CACHE[w] = result
    return result

def translate_words(words: List[str]) -> List[str]:
    return [translate_word(w) for w in words if str(w).strip()]


# ── Low-style meaning enrichment helpers ──────────────────────────────────────
_EN_STOP = {
    'a','an','the','and','or','but','if','then','else','to','of','in','on','at','by','for','from',
    'with','about','as','is','are','was','were','be','been','being','do','does','did','doing',
    'have','has','had','having','that','this','these','those','it','its','he','she','they','them',
    'his','her','their','we','you','i','me','my','our','your','just','very','so','than','too',
    'into','onto','over','under','after','before','because','what','which','who','whom','where',
    'when','why','how'
}

def _norm_en_token(text: str) -> str:
    return re.sub(r"[^a-zA-Z'-]", "", str(text).lower()).strip(" -'")

def _tokenize_en(text: str) -> List[str]:
    toks = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", str(text))
    return [_norm_en_token(t) for t in toks if _norm_en_token(t)]

def _context_window(sentence: str, pivot: str, window: int = 4) -> List[str]:
    toks = _tokenize_en(sentence)
    pivot_n = _norm_en_token(pivot)
    if not toks or not pivot_n:
        return []

    idxs = [i for i, t in enumerate(toks) if t == pivot_n]
    if not idxs:
        idxs = [i for i, t in enumerate(toks) if pivot_n in t or t in pivot_n]

    if not idxs:
        content = [t for t in toks if t not in _EN_STOP and len(t) >= 3 and t != pivot_n]
        return content[:window]

    i = idxs[0]
    left = toks[max(0, i - window):i]
    right = toks[i + 1:i + 1 + window]
    out = []
    for t in left + right:
        if t not in _EN_STOP and len(t) >= 3 and t != pivot_n:
            out.append(t)
    return out[:window]

def _wordnet_related_en(word: str, limit: int = 6) -> List[str]:
    word = _norm_en_token(word)
    if not word:
        return []

    out = []
    seen = {word}
    try:
        synsets = wn.synsets(word)
        for ss in synsets[:6]:
            for lem in ss.lemmas():
                name = _norm_en_token(lem.name().replace('_', ' '))
                if not name or name in seen or ' ' in name:
                    continue
                if len(name) < 3:
                    continue
                out.append(name)
                seen.add(name)
                if len(out) >= limit:
                    return out
    except Exception:
        pass
    return out

def build_low_meanings(sentence: str, pun_word: str, alt_word: str) -> Dict[str, List[str]]:
    """
    Build richer semantic cue lists for Low-style translation.
    meaning1 = pun word + nearby context + WordNet neighbors of pun word
    meaning2 = alternative reading + nearby context + WordNet neighbors of alternative
    """
    pun_word = "" if pun_word is None else str(pun_word).strip()
    alt_word = "" if alt_word is None else str(alt_word).strip()
    sentence = "" if sentence is None else str(sentence)

    ctx_from_pun = _context_window(sentence, pun_word, window=4)
    ctx_from_alt = _context_window(sentence, alt_word if alt_word else pun_word, window=4)

    meaning1 = []
    meaning2 = []

    if pun_word:
        meaning1.append(_norm_en_token(pun_word))
    meaning1.extend(ctx_from_pun)
    meaning1.extend(_wordnet_related_en(pun_word, limit=5))

    if alt_word:
        meaning2.append(_norm_en_token(alt_word))
        meaning2.extend(_wordnet_related_en(alt_word, limit=5))
    else:
        meaning2.extend(_wordnet_related_en(pun_word, limit=5))
    meaning2.extend(ctx_from_alt)

    # de-duplicate while preserving order
    def _dedupe(xs):
        out, seen = [], set()
        for x in xs:
            x = _norm_en_token(x)
            if not x or x in seen:
                continue
            seen.add(x)
            out.append(x)
        return out

    meaning1 = _dedupe(meaning1)
    meaning2 = _dedupe(meaning2)

    if not meaning1 and pun_word:
        meaning1 = [_norm_en_token(pun_word)]
    if not meaning2 and alt_word:
        meaning2 = [_norm_en_token(alt_word)]
    if not meaning2 and pun_word:
        meaning2 = [_norm_en_token(pun_word)]

    return {
        'meaning1_terms': meaning1,
        'meaning2_terms': meaning2,
        'meaning1_gloss': " ".join(meaning1[:5]),
        'meaning2_gloss': " ".join(meaning2[:5]),
    }


# ── Lexique homophone index ────────────────────────────────────────────────────
class LexiquePhoneticIndex:
    def __init__(self, lexique_path: str):
        if not os.path.exists(lexique_path):
            raise FileNotFoundError(
                f"Lexique not found: {lexique_path}\n"
                "Download from http://lexique.org/#openlexicon"
            )
        df = self._load(lexique_path)
        cols = {str(c).lower(): c for c in df.columns}
        ortho = cols.get('ortho') or cols.get('word') or list(df.columns)[0]
        phon  = cols.get('phon') or cols.get('phonology') or cols.get('ipa')
        if phon is None:
            raise ValueError(f"No phonetic column found. Columns: {list(df.columns)[:10]}")
        self.word_to_phon: Dict[str, str] = {}
        self.phon_to_words: Dict[str, List[str]] = {}
        for _, row in df[[ortho, phon]].dropna().iterrows():
            w, p = str(row[ortho]).strip().lower(), str(row[phon]).strip()
            if not w or not p:
                continue
            self.word_to_phon[w] = p
            self.phon_to_words.setdefault(p, []).append(w)
        print(f"  Lexique loaded: {len(self.word_to_phon):,} words")

    @staticmethod
    def _load(path):
        for sep in ['\t', ';', ',', None]:
            try:
                df = pd.read_csv(path, sep=sep, engine='python', on_bad_lines='skip')
                if df is not None and len(df.columns) >= 2 and len(df) > 0:
                    return df
            except Exception:
                pass
        raise ValueError("Could not parse Lexique file.")

    def find_homophones(self, word: str, limit: int = 30) -> List[str]:
        p = self.word_to_phon.get(word.lower().strip())
        if not p:
            return []
        return [w for w in self.phon_to_words.get(p, []) if w != word.lower()][:limit]


# ── WordNet semantic similarity ────────────────────────────────────────────────
class WordNetSemantic:
    def __init__(self):
        self._cache: Dict[Tuple, float] = {}

    def _synsets(self, word: str):
        try:
            ss = wn.synsets(word, lang='fra')
            return ss if ss else wn.synsets(word, lang='eng')
        except Exception:
            return []

    def similarity(self, w1: str, w2: str) -> float:
        w1 = str(w1).strip().lower()
        w2 = str(w2).strip().lower()
        if not w1 or not w2:
            return 0.0

        k = (min(w1, w2), max(w1, w2))
        if k in self._cache:
            return self._cache[k]

        ss1, ss2 = self._synsets(w1), self._synsets(w2)
        if not ss1 or not ss2:
            score = difflib.SequenceMatcher(None, w1, w2).ratio() * 0.3
        else:
            best = 0.0
            for s1 in ss1[:5]:
                for s2 in ss2[:5]:
                    try:
                        sim = s1.wup_similarity(s2)
                        if sim and sim > best:
                            best = sim
                    except Exception:
                        pass
            score = best

        self._cache[k] = score
        return score

    def related_map(self, word: str) -> Dict[str, float]:
        related: Dict[str, float] = {}
        for ss in self._synsets(word)[:6]:
            for lem in ss.lemmas():
                n = lem.name().replace('_', ' ').lower()
                related[n] = max(related.get(n, 0.0), 8.0)
            for hyper in ss.hypernyms():
                for lem in hyper.lemmas():
                    n = lem.name().replace('_', ' ').lower()
                    related[n] = max(related.get(n, 0.0), 4.0)
            for hypo in ss.hyponyms():
                for lem in hypo.lemmas():
                    n = lem.name().replace('_', ' ').lower()
                    related[n] = max(related.get(n, 0.0), 3.0)
        return related

In [16]:
# ── Low's Polygon Translator ──────────────────────────────────────────────────
class LowPolygonTranslator:
    """
    Implements a stronger Low-style polygon search.

    Key improvements:
    - accepts richer meaning cue lists, not just one raw string each
    - scores candidates against multiple translated sense cues
    - literal fallback translates the pun word itself, not a meaning gloss
    """

    RELATION_BONUS = {
        'homophone': 0.90,
        'synonym': 0.55,
        'direct': 0.35
    }

    PATTERN_WEIGHTS = {
        'word_pivot':            {'balanced': 0.50, 'strong': 0.30, 'relation': 0.25},
        'phrase_pivot':          {'balanced': 0.50, 'strong': 0.25, 'relation': 0.25},
        'structure_based':       {'balanced': 0.40, 'strong': 0.35, 'relation': 0.25},
        'full_reinterpretation': {'balanced': 0.55, 'strong': 0.25, 'relation': 0.20},
        'unknown':               {'balanced': 0.50, 'strong': 0.30, 'relation': 0.25},
    }

    MIN_SIM = 0.05

    def __init__(self, lexique_path: str):
        self.lexique  = LexiquePhoneticIndex(lexique_path)
        self.semantic = WordNetSemantic()

    def _norm(self, t: str) -> str:
        return re.sub(r"[^\w' -]", '', str(t).lower().strip())

    def _tokenize_fr(self, text: str) -> List[str]:
        text = self._norm(text)
        if not text:
            return []
        parts = re.findall(r"[a-zàâçéèêëîïôûùüÿñæœ'-]+", text, flags=re.I)
        out = []
        for p in parts:
            p = p.strip(" -'")
            if len(p) >= 2:
                out.append(p.lower())
        return out

    def _prepare_targets(self, meaning) -> List[str]:
        """
        Translate cue words / glosses into FR target terms for scoring.
        Accepts either a string or a list of strings.
        """
        if meaning is None:
            return []

        if isinstance(meaning, (list, tuple, set)):
            items = [str(x).strip() for x in meaning if str(x).strip()]
        else:
            items = [str(meaning).strip()] if str(meaning).strip() else []

        out = []
        seen = set()

        for item in items:
            fr = translate_word(item)
            candidates = [fr] + self._tokenize_fr(fr)
            for c in candidates:
                c = self._norm(c)
                if not c or c in seen:
                    continue
                seen.add(c)
                out.append(c)

        return out

    def _english_seed_words(self, meaning) -> List[str]:
        if meaning is None:
            return []
        if isinstance(meaning, (list, tuple, set)):
            raw = [str(x).strip().lower() for x in meaning]
        else:
            raw = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", str(meaning).lower())

        out = []
        seen = set()
        for r in raw:
            r = re.sub(r"[^a-zA-Z'-]", "", r).strip(" -'")
            if not r or len(r) < 3 or r in seen:
                continue
            seen.add(r)
            out.append(r)
        return out

    def _low_score(self, sim1, sim2, pattern, relation, cand, base):
        w = self.PATTERN_WEIGHTS.get(pattern, self.PATTERN_WEIGHTS['unknown'])
        rel = self.RELATION_BONUS.get(relation, 0.35)
        penalty = 0.10 if cand == base else 0.0
        return (
            w['balanced'] * min(sim1, sim2) +
            w['strong']   * max(sim1, sim2) +
            w['relation'] * rel -
            penalty
        )

    def _score(self, cand, base_fr, rtype, t1s, t2s, pattern='unknown'):
        s1 = max((self.semantic.similarity(cand, t) for t in t1s), default=0.0)
        s2 = max((self.semantic.similarity(cand, t) for t in t2s), default=0.0)
        return {
            'candidate': cand,
            'base_fr': base_fr,
            'relation_type': rtype,
            'sim1': s1,
            'sim2': s2,
            'low_score': self._low_score(s1, s2, pattern, rtype, cand, base_fr)
        }

    def _fr_synonyms(self, fr_word: str, min_w: float = 2.5, limit: int = 12) -> List[str]:
        base = self._norm(fr_word)
        fr_vocab = set(self.lexique.word_to_phon.keys())
        out, seen = [], {base}

        for w, score in sorted(self.semantic.related_map(base).items(), key=lambda x: -x[1]):
            w = self._norm(w)
            if not w or w in seen:
                continue
            if ' ' in w:
                continue
            if score < min_w:
                continue
            if w not in fr_vocab:
                continue
            out.append(w)
            seen.add(w)
            if len(out) >= limit:
                break
        return out

    def _square(self, pun_word, meaning1, meaning2):
        t1s = self._prepare_targets(meaning1)
        t2s = self._prepare_targets(meaning2)

        base_fr = self._norm(translate_word(pun_word))
        if not base_fr:
            return None

        scored, seen = [], set()
        candidates = (
            [(base_fr, 'direct')] +
            [(s, 'synonym') for s in self._fr_synonyms(base_fr)] +
            [(h, 'homophone') for h in self.lexique.find_homophones(base_fr)[:20]]
        )

        for cand, rtype in candidates:
            cand = self._norm(cand)
            if not cand:
                continue
            key = (cand, base_fr, rtype)
            if key in seen:
                continue
            seen.add(key)
            scored.append(self._score(cand, base_fr, rtype, t1s, t2s))

        if not scored:
            return None

        scored.sort(key=lambda x: -x['low_score'])
        best = scored[0]

        if max(best['sim1'], best['sim2']) < self.MIN_SIM:
            return None

        return TranslationCandidate(
            pun_word=best['candidate'],
            polygon_level=4,
            path=[pun_word, best['base_fr'], best['relation_type'], best['candidate']],
            explanation='Square: direct FR translation + synonym/homophone family + Low scoring',
            confidence=min(1.0, best['low_score'])
        )

    def _pentagon(self, pun_word, meaning1, meaning2):
        t1s = self._prepare_targets(meaning1)
        t2s = self._prepare_targets(meaning2)

        base_fr = self._norm(translate_word(pun_word))
        if not base_fr:
            return None

        scored, seen = [], set()

        for syn in self._fr_synonyms(base_fr):
            for h in self.lexique.find_homophones(syn)[:12]:
                h = self._norm(h)
                key = (h, base_fr, 'homophone')
                if not h or key in seen:
                    continue
                seen.add(key)
                scored.append(self._score(h, base_fr, 'homophone', t1s, t2s))

        for h in self.lexique.find_homophones(base_fr)[:20]:
            h = self._norm(h)
            key = (h, base_fr, 'homophone')
            if not h or key in seen:
                continue
            seen.add(key)
            scored.append(self._score(h, base_fr, 'homophone', t1s, t2s))

        if not scored:
            return None

        scored.sort(key=lambda x: -x['low_score'])
        best = scored[0]

        if max(best['sim1'], best['sim2']) < self.MIN_SIM:
            return None

        return TranslationCandidate(
            pun_word=best['candidate'],
            polygon_level=5,
            path=[pun_word, base_fr, 'synonym/homophone expansion', best['candidate']],
            explanation='Pentagon: homophones of FR synonyms, then Low re-ranking',
            confidence=min(1.0, best['low_score'])
        )

    def _hexagon(self, pun_word, meaning1, meaning2):
        """
        Hexagon:
        EN semantic seeds -> EN synonyms -> FR translations -> FR homophones -> Low scoring
        """
        t1s = self._prepare_targets(meaning1)
        t2s = self._prepare_targets(meaning2)

        seeds = []
        for s in self._english_seed_words(meaning1) + self._english_seed_words(meaning2):
            if s not in seeds:
                seeds.append(s)

        if not seeds:
            seeds = [self._norm(pun_word)]

        en_syns = []
        seen_syns = set()

        for seed in seeds[:8]:
            if seed not in seen_syns:
                en_syns.append(seed)
                seen_syns.add(seed)
            try:
                for ss in wn.synsets(seed)[:4]:
                    for lem in ss.lemmas():
                        n = lem.name().replace('_', ' ').lower()
                        n = re.sub(r"[^a-zA-Z' -]", "", n).strip()
                        if not n or ' ' in n or len(n) < 3 or n in seen_syns:
                            continue
                        seen_syns.add(n)
                        en_syns.append(n)
                        if len(en_syns) >= 16:
                            break
                    if len(en_syns) >= 16:
                        break
            except Exception:
                pass
            if len(en_syns) >= 16:
                break

        scored, seen = [], set()

        for syn in en_syns:
            fr = self._norm(translate_word(syn))
            if not fr:
                continue

            # score the translated synonym itself
            key = (fr, fr, 'direct')
            if key not in seen:
                seen.add(key)
                scored.append(self._score(fr, fr, 'direct', t1s, t2s))

            # and score its FR homophones
            for homo in self.lexique.find_homophones(fr)[:12]:
                homo = self._norm(homo)
                key = (homo, fr, 'homophone')
                if not homo or key in seen:
                    continue
                seen.add(key)
                scored.append(self._score(homo, fr, 'homophone', t1s, t2s))

        if not scored:
            return None

        scored.sort(key=lambda x: -x['low_score'])
        best = scored[0]

        if max(best['sim1'], best['sim2']) < self.MIN_SIM:
            return None

        return TranslationCandidate(
            pun_word=best['candidate'],
            polygon_level=6,
            path=[pun_word, 'EN semantic expansion', best['base_fr'], best['candidate']],
            explanation='Hexagon: EN semantic expansion → FR translation → FR homophone → Low scoring',
            confidence=min(1.0, best['low_score'])
        )

    def translate_pun(self, pun_word: str, meaning1, meaning2,
                      max_polygon: int = 6) -> Tuple[Optional[TranslationCandidate],
                                                     Optional[FallbackTranslation]]:
        """
        Try square -> pentagon -> hexagon.
        Fall back to literal pun-word translation.
        """
        for level, method in [
            (4, lambda: self._square(pun_word, meaning1, meaning2)),
            (5, lambda: self._pentagon(pun_word, meaning1, meaning2)),
            (6, lambda: self._hexagon(pun_word, meaning1, meaning2)),
        ]:
            if level > max_polygon:
                break
            result = method()
            if result:
                return result, None

        literal = translate_word(pun_word)
        return None, FallbackTranslation(
            strategy='Literal',
            translation=literal,
            explanation=f"No polygon solution. Literal pun-word translation: '{pun_word}' → '{literal}'"
        )

print("✅ LowPolygonTranslator defined")

✅ LowPolygonTranslator defined


In [17]:
# ── Initialise translator ──────────────────────────────────────────────────────
try:
    poly = LowPolygonTranslator(LEXIQUE_PATH)
    lexique_ok = True
    print(f"✅ Polygon translator ready (target: {TARGET_LANG})")
except FileNotFoundError as e:
    print(f"⚠️  {e}")
    print("Step 4 will record errors but skip polygon translation.")
    lexique_ok = False
    poly = None


  Lexique loaded: 125,652 words
✅ Polygon translator ready (target: fr)


In [18]:
# ── Run Step 4 on evaluation sample ───────────────────────────────────────────
print(f"Running Step 4 translations on {len(df_eval)} rows…")
print("(First call may download the argostranslate language pack)\n")

step4_rows = []

for i, row in df_step3.iterrows():
    pun_word = str(row.get('predicted_location', '') or '').strip()
    alt_word = str(row.get('predicted_alternative', '') or '').strip()
    pun_type = str(row.get('predicted_type', '') or '').strip()
    sentence = str(row.get('text_clean', '') or '').strip()

    result = {
        'id_en': row['id_en'],
        'pun_word': pun_word,
        'pun_word_translated': '',
        'alternative_word': alt_word,
        'alternative_translated': '',
        'target_lang': TARGET_LANG,
        'polygon_level': -1,
        'translation_error': '',
        'meaning1_used': '',
        'meaning2_used': '',
        'translation_strategy': '',
    }

    if not pun_word:
        result['translation_error'] = 'no pun word predicted'
        step4_rows.append(result)
        continue

    try:
        # Build richer Low-style semantic inputs from sentence context.
        meaning_bundle = build_low_meanings(sentence, pun_word, alt_word)
        meaning1_terms = meaning_bundle['meaning1_terms']
        meaning2_terms = meaning_bundle['meaning2_terms']

        result['meaning1_used'] = " | ".join(meaning1_terms)
        result['meaning2_used'] = " | ".join(meaning2_terms)

        # Polygon search only makes sense for French here because Lexique383 is French.
        use_polygon = (TARGET_LANG == 'fr' and lexique_ok and poly is not None)

        if use_polygon:
            candidate, fallback = poly.translate_pun(
                pun_word=pun_word,
                meaning1=meaning1_terms,
                meaning2=meaning2_terms,
            )

            if candidate is not None:
                result['pun_word_translated'] = candidate.pun_word
                result['polygon_level'] = candidate.polygon_level
                result['translation_strategy'] = f'low_polygon_{candidate.polygon_level}'
            else:
                result['pun_word_translated'] = fallback.translation
                result['polygon_level'] = 0
                result['translation_strategy'] = 'literal_fallback'

            if alt_word:
                result['alternative_translated'] = translate_word(alt_word)

        else:
            # Literal fallback is still a valid translation, not an error.
            result['pun_word_translated'] = translate_word(pun_word)
            result['alternative_translated'] = translate_word(alt_word) if alt_word else ''
            result['polygon_level'] = 0
            result['translation_strategy'] = 'literal_only'

        # Treat empty output as a real failure.
        if not result['pun_word_translated']:
            result['translation_error'] = 'empty translation output'

    except Exception as exc:
        result['translation_error'] = str(exc)

    step4_rows.append(result)

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(df_eval)} done…")

df_step4 = pd.DataFrame(step4_rows)
step4_out = os.path.join(OUTPUT_DIR, "step4_translations.tsv")
df_step4.to_csv(step4_out, sep="\t", index=False)

print(f"\n✅ Saved → {step4_out}")
df_step4.head(5)

Running Step 4 translations on 100 rows…
(First call may download the argostranslate language pack)



INFO:argostranslate.utils:('get_installed_languages',)
INFO:argostranslate.utils:('paragraphs:', ['catch'])
INFO:argostranslate.utils:('apply_packaged_translation', 'catch')
INFO:argostranslate.utils:('Splitting sentences using SBD Model: (en) StanzaSentencizer',)
INFO:argostranslate.utils:('sentences', ['catch'])
INFO:argostranslate.utils:('tokenized', [['▁catch']])
INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁prise']], scores=[-1.805282711982727], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('prise', -1.805282711982727)])
INFO:argostranslate.utils:('translated_paragraphs:', [[('prise', -1.805282711982727)]])
INFO:argostranslate.utils:('hypotheses_to_return:', [('prise', -1.805282711982727)])
INFO:argostranslate.utils:('paragraphs:', ['see'])
INFO:argostranslate.utils:('apply_packaged_translation', 'see')
INFO:argostranslate.utils:('Splitting sentences using SBD Model: (en) StanzaSentencizer',)
INFO:argostranslate.u

  10/100 done…


INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁élément']], scores=[-0.2930101454257965], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('élément', -0.2930101454257965)])
INFO:argostranslate.utils:('translated_paragraphs:', [[('élément', -0.2930101454257965)]])
INFO:argostranslate.utils:('hypotheses_to_return:', [('élément', -0.2930101454257965)])
INFO:argostranslate.utils:('paragraphs:', ['surprisal'])
INFO:argostranslate.utils:('apply_packaged_translation', 'surprisal')
INFO:argostranslate.utils:('Splitting sentences using SBD Model: (en) StanzaSentencizer',)
INFO:argostranslate.utils:('sentences', ['surprisal'])
INFO:argostranslate.utils:('tokenized', [['▁sur', 'pri', 's', 'al']])
INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁Sur', 'pri', 's', 'al']], scores=[-2.647069215774536], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('Surprisal', -2.64706921577

  20/100 done…


...

INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁sa', 'liv', 'aire']], scores=[-1.5921919345855713], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('salivaire', -1.5921919345855713)])
INFO:argostranslate.utils:('translated_paragraphs:', [[('salivaire', -1.5921919345855713)]])
INFO:argostranslate.utils:('hypotheses_to_return:', [('salivaire', -1.5921919345855713)])
INFO:argostranslate.utils:('paragraphs:', ['glands'])
INFO:argostranslate.utils:('apply_packaged_translation', 'glands')
INFO:argostranslate.utils:('Splitting sentences using SBD Model: (en) StanzaSentencizer',)
INFO:argostranslate.utils:('sentences', ['glands'])
INFO:argostranslate.utils:('tokenized', [['▁gland', 's']])
INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁glande', 's']], scores=[-0.9017915725708008], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('glandes', -0.9017915725708008)])
INFO:a

  90/100 done…


INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁entier']], scores=[-0.6150942444801331], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('entier', -0.6150942444801331)])
INFO:argostranslate.utils:('translated_paragraphs:', [[('entier', -0.6150942444801331)]])
INFO:argostranslate.utils:('hypotheses_to_return:', [('entier', -0.6150942444801331)])
INFO:argostranslate.utils:('paragraphs:', ['thing'])
INFO:argostranslate.utils:('apply_packaged_translation', 'thing')
INFO:argostranslate.utils:('Splitting sentences using SBD Model: (en) StanzaSentencizer',)
INFO:argostranslate.utils:('sentences', ['thing'])
INFO:argostranslate.utils:('tokenized', [['▁thing']])
INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁une', '▁chose']], scores=[-2.7663424015045166], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('une chose', -2.7663424015045166)])
INFO:argostranslate.utils:('tr

  100/100 done…

✅ Saved → outputs/step4_translations.tsv


,id_en,pun_word,pun_word_translated,alternative_word,alternative_translated,target_lang,polygon_level,translation_error,meaning1_used,meaning2_used,translation_strategy
0,en_6789,catch,confiscation,catch,prise,fr,4,,catch | see | gimmick | haul | match | stop,catch | gimmick | haul | match | stop | see,low_polygon_4
1,en_3677,pin,pin,pin,broche,fr,4,,pin | brooch | couldn't | fall | peg | persona...,pin | fall | peg | personalidentificationnumbe...,low_polygon_4
2,en_4418,presence,mien,presence,présence,fr,4,,presence | interested | front | bearing | comp...,presence | front | bearing | comportment | mie...,low_polygon_4
3,en_4428,hopefully,espérons,hopeful,Espérons,fr,4,,hopefully | big | diamond | said,hopeful | aspirant | aspirer | wannabe | wanna...,low_polygon_4
4,en_6161,cougher,tousseur,cougher,tousseurs,fr,4,,cougher | made | couldn't | refuse,cougher | made | couldn't | refuse,low_polygon_4


In [19]:
# ── evaluator.py — evaluate_translations ──────────────────────────────────────
# Uses cosine similarity between sentence-transformer embeddings.
# Since HuggingFace may be blocked in some environments, we fall back to
# scipy cosine on TF-IDF vectors if the model cannot be downloaded.

from sklearn.feature_extraction.text import TfidfVectorizer

def evaluate_translations(df: pd.DataFrame) -> dict:
    """
    Mirrors evaluator.py evaluate_translations.
    Metrics: mean cosine similarity, variance, Q1, Q3, error count.

    Computes cosine similarity between (pun_word, pun_word_translated) pairs
    using TF-IDF character n-gram vectors as an offline proxy for semantic
    similarity. In production, swap tfidf_cosine for sentence-transformer cosine.
    """
    errors = df[df['translation_error'] != '']
    valid  = df[df['translation_error'] == ''].copy()

    if len(valid) == 0:
        return {'mean_cosine': 0.0, 'variance': 0.0, 'q1': 0.0, 'q3': 0.0,
                'error_count': len(errors), 'valid_count': 0}

    # Attempt sentence-transformers (if model cached)
    sims = []
    try:
        from sentence_transformers import SentenceTransformer, util
        model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
        src_emb = model.encode(valid['pun_word'].tolist(), convert_to_tensor=True)
        tgt_emb = model.encode(valid['pun_word_translated'].tolist(), convert_to_tensor=True)
        sims    = util.cos_sim(src_emb, tgt_emb).diagonal().cpu().numpy().tolist()
        print("  Using sentence-transformer cosine similarity")
    except Exception:
        # Fallback: TF-IDF character n-gram cosine (offline, always works)
        print("  Using TF-IDF character n-gram cosine (offline fallback)")
        all_words = valid['pun_word'].tolist() + valid['pun_word_translated'].tolist()
        vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
        vec.fit(all_words)
        n = len(valid)
        src_vecs = vec.transform(valid['pun_word'].tolist())
        tgt_vecs = vec.transform(valid['pun_word_translated'].tolist())
        for i in range(n):
            s = src_vecs[i].toarray().flatten()
            t = tgt_vecs[i].toarray().flatten()
            if np.linalg.norm(s) == 0 or np.linalg.norm(t) == 0:
                sims.append(0.0)
            else:
                sims.append(float(1 - spdist.cosine(s, t)))

    sims_arr = np.array(sims)
    return {
        'mean_cosine': float(np.mean(sims_arr)),
        'variance':    float(np.var(sims_arr)),
        'q1':          float(np.percentile(sims_arr, 25)),
        'q3':          float(np.percentile(sims_arr, 75)),
        'error_count': int(len(errors)),
        'valid_count': int(len(valid)),
    }

metrics_s4 = evaluate_translations(df_step4)

print("\n" + "═"*55)
print("STEP 4 RESULTS — Translation Evaluation")
print("═"*55)
print(f"  Mean cosine similarity : {metrics_s4['mean_cosine']:.3f}")
print(f"  Variance               : {metrics_s4['variance']:.4f}")
print(f"  Q1 (bottom quartile)   : {metrics_s4['q1']:.3f}")
print(f"  Q3 (top quartile)      : {metrics_s4['q3']:.3f}")
print(f"  Errors                 : {metrics_s4['error_count']}")
print(f"  Valid translations     : {metrics_s4['valid_count']}")
print("═"*55)

# Polygon level distribution
if lexique_ok:
    lvl_counts = df_step4['polygon_level'].value_counts().sort_index()
    print("\nPolygon level distribution:")
    lvl_labels = {-1:'no pred', 0:'fallback', 4:'square', 5:'pentagon', 6:'hexagon'}
    for lvl, cnt in lvl_counts.items():
        print(f"  {lvl_labels.get(int(lvl), str(lvl)):<12}: {cnt}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Using sentence-transformer cosine similarity

═══════════════════════════════════════════════════════
STEP 4 RESULTS — Translation Evaluation
═══════════════════════════════════════════════════════
  Mean cosine similarity : 0.426
  Variance               : 0.1112
  Q1 (bottom quartile)   : 0.158
  Q3 (top quartile)      : 0.697
  Errors                 : 0
  Valid translations     : 100
═══════════════════════════════════════════════════════

Polygon level distribution:
  square      : 100


## 8. Full Results Summary

In [20]:
print("\n" + "█"*65)
print("  FULL PIPELINE EVALUATION SUMMARY")
print("  Dataset: " + TSV_PATH)
print(f"  Eval sample: {len(df_eval)} rows | Target language: {TARGET_LANG}")
print("█"*65)

print("\n┌─────────────────────────────────────────────────────────────┐")
print("│  STEP 1 — Pun Word Identification (GBM token classifier)    │")
print("├─────────────────────────────────────────────────────────────┤")
print(f"│  Accuracy  : {metrics_s1['accuracy']:.3f}  ({metrics_s1['correct']}/{metrics_s1['total']} correct)               │")
print(f"│  Precision : {metrics_s1['precision']:.3f}                                        │")
print(f"│  Recall    : {metrics_s1['recall']:.3f}                                        │")
print(f"│  F1        : {metrics_s1['f1']:.3f}                                        │")
print("├─────────────────────────────────────────────────────────────┤")
print("│  STEP 2 — Pun Type Classification (Logistic Regression)     │")
print("├─────────────────────────────────────────────────────────────┤")
print(f"│  Accuracy  : {metrics_s2['accuracy']:.3f}                                        │")
print(f"│  Precision : {metrics_s2['precision']:.3f} (macro)                               │")
print(f"│  Recall    : {metrics_s2['recall']:.3f} (macro)                               │")
print(f"│  F1        : {metrics_s2['f1']:.3f} (macro)                               │")
print("├─────────────────────────────────────────────────────────────┤")
print("│  STEP 3 — Alternative Meaning (Metaphone lookup)            │")
print("├─────────────────────────────────────────────────────────────┤")
print(f"│  Accuracy  : {metrics_s3['accuracy']:.3f}  ({metrics_s3['correct']}/{metrics_s3['total']} correct)               │")
print(f"│  Precision : {metrics_s3['precision']:.3f}                                        │")
print(f"│  Recall    : {metrics_s3['recall']:.3f}                                        │")
print(f"│  F1        : {metrics_s3['f1']:.3f}                                        │")
print("├─────────────────────────────────────────────────────────────┤")
print("│  STEP 4 — Translation (Low's Polygon Algorithm)             │")
print("├─────────────────────────────────────────────────────────────┤")
print(f"│  Mean cosine similarity : {metrics_s4['mean_cosine']:.3f}                        │")
print(f"│  Variance               : {metrics_s4['variance']:.4f}                       │")
print(f"│  Q1 / Q3                : {metrics_s4['q1']:.3f} / {metrics_s4['q3']:.3f}                    │")
print(f"│  Errors                 : {metrics_s4['error_count']}                                     │")
print("└─────────────────────────────────────────────────────────────┘")

print("\nOutput files:")
for f in [step1_out, step2_out, step3_out, step4_out]:
    n = len(pd.read_csv(f, sep='\t'))
    print(f"  {f}  ({n} rows)")

print("\n✅ To run with evaluator.py:")
print("""
  from evaluator import (evaluate_pun_location, evaluate_pun_type,
                          evaluate_alternative_words, evaluate_translations)

  evaluate_pun_location(    pd.read_csv('outputs/step1_pun_location.tsv',  sep='\\t'))
  evaluate_pun_type(        pd.read_csv('outputs/step2_pun_type.tsv',      sep='\\t'))
  evaluate_alternative_words(pd.read_csv('outputs/step3_alternative.tsv',  sep='\\t'))
  evaluate_translations(    pd.read_csv('outputs/step4_translations.tsv',  sep='\\t'))
""")



█████████████████████████████████████████████████████████████████
  FULL PIPELINE EVALUATION SUMMARY
  Dataset: combined_en_train.tsv
  Eval sample: 100 rows | Target language: fr
█████████████████████████████████████████████████████████████████

┌─────────────────────────────────────────────────────────────┐
│  STEP 1 — Pun Word Identification (GBM token classifier)    │
├─────────────────────────────────────────────────────────────┤
│  Accuracy  : 0.900  (90/100 correct)               │
│  Precision : 1.000                                        │
│  Recall    : 0.900                                        │
│  F1        : 0.947                                        │
├─────────────────────────────────────────────────────────────┤
│  STEP 2 — Pun Type Classification (Logistic Regression)     │
├─────────────────────────────────────────────────────────────┤
│  Accuracy  : 0.760                                        │
│  Precision : 0.764 (macro)                               │
│  R